In [ ]:
# %% Deep learning - Section 19.184
#    Do autoencoders clean Gaussians ?

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [2]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [4]:
# %% Data

# Load data
# See also : https://www.nist.gov/itl/products-and-services/emnist-dataset
data = torchvision.datasets.EMNIST(root='emnist',split='letters',download=True)


In [ ]:
# Inspect the data

# Categories (27 because extra n/a)
print(data.classes)
print(str(len(data.classes)) + ' classes')

# Original is 3D
print('\nData size:')
print(data.data.shape)

# Transform into 4D tensor for convolution layers, transform from int8 to float
images = data.data.view([124800,1,28,28]).float()
print('\nTensor data:')
print(images.shape)


In [ ]:
# %% N/A issue

# The class N/A doesn't exist in the data
print( torch.sum(data.targets==0) )

# However, it causes problems in one-hot encoding
print(torch.unique(data.targets))
data.class_to_idx


In [ ]:
# Remove N/A

# Remove the first class category
letter_categories = data.classes[1:]

# relabel labels to start at 0
labels = copy.deepcopy(data.targets)-1
print(labels.shape)
print(torch.sum(labels==0))
torch.unique(labels)


In [ ]:
# %% Normalise data

# Before norm
plt.hist(images[:10,:,:,:].flatten().detach(),40);
plt.title('Raw values')
plt.show()

# Norm
images /= torch.max(images)

# After norm
plt.hist(images[:10,:,:,:].flatten().detach(),40);
plt.title('After normalization')
plt.show()


In [ ]:
# Plotting

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,7,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):

    whichpic = np.random.randint(images.shape[0])

    I = np.squeeze( images[whichpic,:,:] )
    letter = letter_categories[labels[whichpic]]

    ax.imshow(I.T,cmap='gray')
    ax.set_title('The letter "%s"'%letter)
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure112_emnist_dataset.png')
plt.show()
files.download('figure112_emnist_dataset.png')


In [13]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [ ]:
# %% Check sizes

print(train_loader.dataset.tensors[0].shape)
print(train_loader.dataset.tensors[1].shape)


In [40]:
# %% Function to generate the model

def gen_model(printing_toggle=False):

    class emnist_CNN(nn.Module):
        def __init__(self,printing_toggle):
            super().__init__()

            # Convolution layer 1
            # size = np.floor( (28+2*1-3)/1 )+1 = 28/2 = 14 (divide by 2 because
            # will have maxpool with extent 2)
            self.conv1  = nn.Conv2d(1,6,kernel_size=3,stride=1,padding=1)

            # Batch normalisation for convolution
            self.bnorm1 = nn.BatchNorm2d(6)

            # Convolution layer 2
            # size = np.floor( (14+2*1-3)/1 )+1 = 14/2 = 7 (divide by 2 because
            # will have maxpool with extent 2)
            self.conv2 = nn.Conv2d(6,6,kernel_size=3,stride=1,padding=1)

            # Batch normalisation for convolution
            self.bnorm2 = nn.BatchNorm2d(6)

            # Fully connected layer with batch normalisation
            self.fc1 = nn.Linear(7*7*6,50)
            self.bnorm_fc1 = nn.BatchNorm1d(50)

            # Output layer
            self.output = nn.Linear(50,26)

            # Toggle for the printing of tensor sizes during forward propagation
            self.print = printing_toggle

        def forward(self,x):

            # Print input layer size
            print(f'Input size: {x.shape}') if self.print else None

            # MaxPool and ReLu on convolution layer 1 (pool before passing to
            # activation function)
            x = F.max_pool2d(self.conv1(x),2)
            x = F.leaky_relu(self.bnorm1(x))
            print(f'Conv. layer 1 size: {x.shape}') if self.print else None

            # MaxPool and ReLu on convolution layer 2
            x = F.max_pool2d(self.conv2(x),2)
            x = F.leaky_relu(self.bnorm2(x))
            print(f'Conv. layer 2 size: {x.shape}') if self.print else None

            # Vectorise for linear layer
            n_units = x.shape.numel() / x.shape[0]
            x       = x.view(-1,int(n_units))
            print(f'Vectorised conv. 2 layer size: {x.shape}') if self.print else None

            # Linear and output layers
            x = F.leaky_relu(self.fc1(x))
            x = self.bnorm_fc1(x)
            print(f'Linear layer size: {x.shape}') if self.print else None
            x = self.output(x)
            print(f'Output layer size: {x.shape}') if self.print else None

            return x

    # Create model instance
    CNN = emnist_CNN(printing_toggle)

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [53]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model()

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and error rate (1-accuracy) from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append(loss.item())
            batch_acc.append( torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))

            # Ship to GPU (y for loss calculation)
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            yHat = yHat.cpu()
            y    = y.cpu()

            test_acc.append( 100*torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )
            test_losses.append(loss.item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [61]:
# %% Fit model

# Takes ~2 mins on GPU
train_err,test_err,train_losses,test_losses,CNN = train_model()


In [ ]:
# Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(train_losses,'s-',label='Train')
ax[0].plot(test_losses,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')

ax[1].plot(train_err,'s-',label='Train')
ax[1].plot(test_err,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Error rates (%)')
ax[1].set_title(f'Final model test error rate: {test_err[-1]:.2f}%')
ax[1].legend()

plt.savefig('figure113_emnist_dataset.png')
plt.show()
files.download('figure113_emnist_dataset.png')


In [ ]:
# %% Plotting

# get data from test dataloader
X,y  = next(iter(test_loader))
X    = X.to(device)
y    = y.to(device)
yHat = CNN(X)

# Pick random examples
rand_id = np.random.choice(len(y),size=21,replace=False)

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,7,figsize=(1.5*phi*6,6))

for i,ax in enumerate(axs.flatten()):

    # Extract image and its target letter; .cpu() to transfer back from GPU
    I = np.squeeze( X[rand_id[i],0,:,:] ).cpu()
    true_letter = letter_categories[ y[rand_id[i]] ]
    pred_letter = letter_categories[ torch.argmax(yHat[rand_id[i],:]) ]

    # Colour-code wrong classification (with ternary operator)
    col = 'gray' if true_letter==pred_letter else 'hot'

    ax.imshow(I.T,cmap=col)
    ax.set_title('True %s, predicted %s' %(true_letter,pred_letter),fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.savefig('figure114_emnist_dataset.png')
plt.show()
files.download('figure114_emnist_dataset.png')


In [ ]:
# %% So many classes! Compute the confusion matrix

# Confusion matrix
C = skm.confusion_matrix(y.cpu(),torch.argmax(yHat.cpu(),axis=1),normalize='true')

# Plot
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(7*phi,7))
plt.imshow(C,'Blues',vmax=.05)

# make the plot look nicer
plt.xticks(range(26),labels=letter_categories)
plt.yticks(range(26),labels=letter_categories)
plt.title('Confusion matrix\nTest data')
plt.xlabel('True letter')
plt.xlabel('Predicted letter')
plt.ylabel('True number')
plt.colorbar()

plt.savefig('figure116_emnist_dataset.png')
plt.show()
files.download('figure116_emnist_dataset.png')


In [ ]:
# %% Exercise 1
#    I added batch normalization to the convolution layers, but not to the linear (fc*) layers. But linear layers also
#    benefit from batchnorm just like convolution layers do. Add it!

# Easily done by adding a few lines in the model class, the performance seems
# roughly the same as without batch norm on the fully connected layer


In [60]:
# %% Exercise 2
#    In the next few videos, we will see whether we can improve the model's performance by experimenting with the number
#    of layers, kernel size, and linear-layer units. Is there anything you could think of, other than these three features,
#    that might help boost model performance?

# By spying the titles of the next video, I would say some regularisation
# techniques like dropout and weight regularisation; here I tried L2
# regularisation. Interestingly, the performance seems to degree with high
# weight decay values (0.01), and it says pretty much the same as no decay for
# smaller values (0.0001)

# Function to generate the model
def gen_model(printing_toggle=False):

    class emnist_CNN(nn.Module):
        def __init__(self,printing_toggle):
            super().__init__()

            self.conv1  = nn.Conv2d(1,6,kernel_size=3,stride=1,padding=1)
            self.bnorm1 = nn.BatchNorm2d(6)

            self.conv2 = nn.Conv2d(6,6,kernel_size=3,stride=1,padding=1)
            self.bnorm2 = nn.BatchNorm2d(6)

            self.fc1 = nn.Linear(7*7*6,50)
            self.bnorm_fc1 = nn.BatchNorm1d(50)

            self.output = nn.Linear(50,26)

            self.print = printing_toggle

        def forward(self,x):

            print(f'Input size: {x.shape}') if self.print else None


            x = F.max_pool2d(self.conv1(x),2)
            x = F.leaky_relu(self.bnorm1(x))
            print(f'Conv. layer 1 size: {x.shape}') if self.print else None

            x = F.max_pool2d(self.conv2(x),2)
            x = F.leaky_relu(self.bnorm2(x))
            print(f'Conv. layer 2 size: {x.shape}') if self.print else None

            n_units = x.shape.numel() / x.shape[0]
            x       = x.view(-1,int(n_units))
            print(f'Vectorised conv. 2 layer size: {x.shape}') if self.print else None

            x = F.leaky_relu(self.fc1(x))
            x = self.bnorm_fc1(x)
            print(f'Linear layer size: {x.shape}') if self.print else None
            x = self.output(x)
            print(f'Output layer size: {x.shape}') if self.print else None

            return x

    CNN       = emnist_CNN(printing_toggle)
    loss_fun  = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001,weight_decay=0.0001)

    return CNN,loss_fun,optimizer
